# Banco de Áreas Elegíveis para Restauração Florestal — PES Charapucu
### Núcleo de Geotecnologias (NGEO) / IDEFLOR-Bio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/restaura-pes-charapucu/blob/main/notebooks/dashboard_charapucu.ipynb)

Este Mini-Dashboard Interativo orquestra a carteira qualificada e auditada de polígonos elegíveis para restauração florestal no **Parque Estadual Charapucu** (Afuá / Arquipélago do Marajó - PA).

**Destaques do Inventário Consolidado:**
* **Meta Básica:** $130{,}00\text{ ha}$
* **Carteira Auditada Final:** $366{,}72\text{ ha}$ (304 glebas — Superávit de **+236,72 ha** / **282,1%** da meta)
* **Métodos:** Enriquecimento ($263{,}24\text{ ha}$), RNA ($94{,}52\text{ ha}$) e Plantio Direto ($8{,}96\text{ ha}$)
* **Logística Fluvial:** $9{,}04\text{ ha}$ em Alta Acessibilidade ($50-200\text{ m}$) e $49{,}04\text{ ha}$ em Média Acessibilidade ($200-800\text{ m}$)

---

In [ ]:
# [1] Instalação de dependências e configuração de ambiente (Google Colab / Local)
import os
import sys

try:
    import google.colab
    IN_COLAB = True
    print('Ambiente Google Colab detectado.')
    !pip install -q geopandas folium plotly ipywidgets shapely
    
    # Se aberto diretamente no Colab pelo GitHub, clona o repositório para ter os dados
    if not os.path.exists('outputs') and not os.path.exists('data'):
        if not os.path.exists('restaura-pes-charapucu'):
            print('Clonando repositório para o ambiente de execução...')
            !git clone https://github.com/NGEO-IDEFLOR-Bio/restaura-pes-charapucu.git
        if os.path.exists('restaura-pes-charapucu'):
            %cd restaura-pes-charapucu
            print('Diretório de trabalho ajustado para:', os.getcwd())
except ImportError:
    IN_COLAB = False
    print('Ambiente Local / Jupyter detectado.')

print('Pronto para execução!')

In [ ]:
# [2] Importação de Bibliotecas e Carga dos Dados Geoespaciais
from pathlib import Path
import json
import geopandas as gpd
import pandas as pd
import folium
from folium import GeoJson, GeoJsonTooltip, GeoJsonPopup
from branca.element import Element
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Identificar caminho dos arquivos (compatível com local e Google Drive/Colab)
caminhos_possiveis_dados = [
    Path("data/processed/banco_restauracao_consolidado.geojson"),
    Path("../data/processed/banco_restauracao_consolidado.geojson"),
    Path("outputs/vetores/banco_restauracao_pes_charapucu.geojson"),
    Path("../outputs/vetores/banco_restauracao_pes_charapucu.geojson")
]

arquivo_dados = next((p for p in caminhos_possiveis_dados if p.exists()), None)
if arquivo_dados is None:
    raise FileNotFoundError("Arquivo geoespacial consolidado não localizado. Execute 'src/03_estratificacao_talhoes.py'.")

print(f"Carregando dados de: {arquivo_dados}...")
gdf_base = gpd.read_file(arquivo_dados)
if gdf_base.crs != "EPSG:4326":
    gdf_base = gdf_base.to_crs("EPSG:4326")

# Carregar limite da UC se disponível
caminhos_limite = [
    Path("data/raw/limite_pes_charapucu.geojson"),
    Path("../data/raw/limite_pes_charapucu.geojson")
]
arquivo_limite = next((p for p in caminhos_limite if p.exists()), None)
gdf_parque = gpd.read_file(arquivo_limite).to_crs("EPSG:4326") if arquivo_limite else None

print(f"✅ {len(gdf_base)} glebas carregadas com sucesso! Área total: {gdf_base['area_ha'].sum():.2f} ha")

In [ ]:
# [3] Funções Construtoras de Cards, Gráficos e Mapa

CORES_METODOS = {
    "RNA": "#2ecc71",           # Verde Esmeralda
    "Enriquecimento": "#f39c12", # Âmbar
    "Plantio Direto": "#e74c3c"  # Vermelho
}

CORES_PORTE = {
    "Pequeno (< 5 ha)": "#3498db",
    "Médio (5 - 20 ha)": "#9b59b6",
    "Grande (20 - 50 ha)": "#e67e22",
    "Macro-gleba (> 50 ha)": "#1abc9c"
}

def gerar_cards_kpi(df_filtrado: pd.DataFrame, meta_ha: float = 130.0) -> str:
    total_ha = float(df_filtrado["area_ha"].sum()) if not df_filtrado.empty else 0.0
    total_talhoes = len(df_filtrado)
    dist_media = float(df_filtrado["dist_margem_m"].mean()) if not df_filtrado.empty else 0.0
    
    if not df_filtrado.empty and "metodo" in df_filtrado.columns:
        metodo_dom = df_filtrado.groupby("metodo")["area_ha"].sum().idxmax()
        area_metodo_dom = df_filtrado.groupby("metodo")["area_ha"].sum().max()
        perc_metodo_dom = (area_metodo_dom / total_ha) * 100.0 if total_ha > 0 else 0
    else:
        metodo_dom = "Nenhum"
        perc_metodo_dom = 0.0

    perc_meta = (total_ha / meta_ha) * 100.0 if meta_ha > 0 else 0.0
    cor_meta = "#27ae60" if perc_meta >= 100 else "#e67e22"

    return f"""
    <div style="display: flex; flex-wrap: wrap; gap: 15px; margin: 15px 0; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif;">
        <div style="flex: 1; min-width: 200px; background: #ffffff; border: 1px solid #e1e8ed; border-left: 5px solid {cor_meta}; border-radius: 8px; padding: 12px 16px; box-shadow: 0 2px 4px rgba(0,0,0,0.04);">
            <div style="font-size: 11px; text-transform: uppercase; font-weight: 700; color: #7f8c8d;">Área Selecionada</div>
            <div style="font-size: 24px; font-weight: 800; color: #2c3e50; margin: 2px 0;">{total_ha:,.2f} <span style="font-size: 13px; font-weight: 600; color: #7f8c8d;">ha</span></div>
            <div style="font-size: 11px; font-weight: 600; color: {cor_meta};">● {perc_meta:.1f}% da meta ({meta_ha:.0f} ha)</div>
        </div>
        <div style="flex: 1; min-width: 200px; background: #ffffff; border: 1px solid #e1e8ed; border-left: 5px solid #2980b9; border-radius: 8px; padding: 12px 16px; box-shadow: 0 2px 4px rgba(0,0,0,0.04);">
            <div style="font-size: 11px; text-transform: uppercase; font-weight: 700; color: #7f8c8d;">Glebas Selecionadas</div>
            <div style="font-size: 24px; font-weight: 800; color: #2c3e50; margin: 2px 0;">{total_talhoes} <span style="font-size: 13px; font-weight: 600; color: #7f8c8d;">unidades</span></div>
            <div style="font-size: 11px; color: #7f8c8d;">Média de {(total_ha/total_talhoes if total_talhoes > 0 else 0):.2f} ha/gleba</div>
        </div>
        <div style="flex: 1; min-width: 200px; background: #ffffff; border: 1px solid #e1e8ed; border-left: 5px solid #16a085; border-radius: 8px; padding: 12px 16px; box-shadow: 0 2px 4px rgba(0,0,0,0.04);">
            <div style="font-size: 11px; text-transform: uppercase; font-weight: 700; color: #7f8c8d;">Distância aos Furos</div>
            <div style="font-size: 24px; font-weight: 800; color: #2c3e50; margin: 2px 0;">{dist_media:,.1f} <span style="font-size: 13px; font-weight: 600; color: #7f8c8d;">m</span></div>
            <div style="font-size: 11px; color: #7f8c8d;">Média até margem navegável</div>
        </div>
        <div style="flex: 1; min-width: 200px; background: #ffffff; border: 1px solid #e1e8ed; border-left: 5px solid #8e44ad; border-radius: 8px; padding: 12px 16px; box-shadow: 0 2px 4px rgba(0,0,0,0.04);">
            <div style="font-size: 11px; text-transform: uppercase; font-weight: 700; color: #7f8c8d;">Método Predominante</div>
            <div style="font-size: 20px; font-weight: 800; color: #2c3e50; margin: 2px 0;">{metodo_dom}</div>
            <div style="font-size: 11px; color: #7f8c8d;">{perc_metodo_dom:.1f}% da área selecionada</div>
        </div>
    </div>
    """

def gerar_graficos_analiticos(df_filtrado: pd.DataFrame) -> go.Figure:
    if df_filtrado.empty:
        fig = go.Figure()
        fig.add_annotation(text="Nenhum talhão atende aos filtros atuais.", showarrow=False, font=dict(size=14))
        fig.update_layout(height=320, margin=dict(l=10, r=10, t=30, b=10))
        return fig

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "domain"}, {"type": "xy"}]],
        subplot_titles=["<b>Área por Método de Restauração</b>", "<b>Distribuição por Classe de Porte</b>"]
    )

    df_m = df_filtrado.groupby("metodo")["area_ha"].sum().reset_index()
    cores_m = [CORES_METODOS.get(m, "#95a5a6") for m in df_m["metodo"]]
    fig.add_trace(
        go.Pie(
            labels=df_m["metodo"], values=df_m["area_ha"], hole=0.55,
            marker=dict(colors=cores_m, line=dict(color="#fff", width=2)),
            textinfo="percent+label", hovertemplate="<b>%{label}</b>: %{value:.2f} ha (%{percent})<extra></extra>"
        ),
        row=1, col=1
    )

    ordem_p = ["Pequeno (< 5 ha)", "Médio (5 - 20 ha)", "Grande (20 - 50 ha)", "Macro-gleba (> 50 ha)"]
    df_p = df_filtrado.groupby("classe_area").agg(area_total=("area_ha", "sum"), qtd=("id_gleba", "count")).reindex(ordem_p).dropna(how="all").reset_index()
    cores_p = [CORES_PORTE.get(p, "#34495e") for p in df_p["classe_area"]]

    fig.add_trace(
        go.Bar(
            x=df_p["classe_area"], y=df_p["area_total"], marker=dict(color=cores_p),
            text=[f"{v:.1f} ha ({q} un)" for v, q in zip(df_p["area_total"], df_p["qtd"])], textposition="auto",
            hovertemplate="<b>%{x}</b><br>Área: %{y:.2f} ha<extra></extra>"
        ),
        row=1, col=2
    )

    fig.update_layout(
        height=320, margin=dict(l=10, r=10, t=40, b=10), showlegend=False,
        plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
        font=dict(family="Segoe UI, Roboto, sans-serif", size=11)
    )
    fig.update_yaxes(title_text="Área (ha)", gridcolor="#edf2f7", row=1, col=2)
    return fig

def construir_mapa(gdf_f: gpd.GeoDataFrame, gdf_p: gpd.GeoDataFrame = None) -> folium.Map:
    m = folium.Map(location=[-0.3414, -50.4998], zoom_start=11, tiles=None, control_scale=True)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery", name="Satélite (Esri World Imagery)", overlay=False, control=True
    ).add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Light Gray", name="Mapa Claro (Esri Light Gray)", overlay=False, control=True
    ).add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Topo Map", name="Topográfico (Esri Topo)", overlay=False, control=True
    ).add_to(m)

    if gdf_p is not None and not gdf_p.empty:
        folium.GeoJson(
            gdf_p, name="Limite Oficial PES Charapucu",
            style_function=lambda x: {"color": "#00ffff", "weight": 2.5, "fillOpacity": 0, "dashArray": "4, 4"},
            tooltip="<b>Parque Estadual Charapucu</b>"
        ).add_to(m)

    if not gdf_f.empty:
        def estilo(f):
            met = f["properties"].get("metodo", "Enriquecimento")
            cor = CORES_METODOS.get(met, "#f39c12")
            return {"color": cor, "weight": 1.5, "fillColor": cor, "fillOpacity": 0.65}

        GeoJson(
            gdf_f, name="Talhões Selecionados", style_function=estilo,
            tooltip=GeoJsonTooltip(
                fields=["id_gleba", "metodo", "area_ha", "dist_margem_m", "acesso_fluvial"],
                aliases=["Gleba:", "Método:", "Área (ha):", "Dist. Furo (m):", "Acesso:"], localize=True
            ),
            popup=GeoJsonPopup(
                fields=["id_gleba", "metodo", "area_ha", "acesso_fluvial", "dist_margem_m", "setor_fluvial"],
                aliases=["ID:", "Método:", "Área (ha):", "Acesso:", "Distância:", "Setor:"], localize=True
            )
        ).add_to(m)

    # Legenda Flutuante
    legenda_html = '''
    <div style="position: fixed; bottom: 25px; left: 25px; width: 220px; background-color: rgba(255, 255, 255, 0.95); border: 2px solid #bdc3c7; border-radius: 8px; z-index: 9999; font-size: 11px; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; padding: 10px 14px; box-shadow: 0 2px 8px rgba(0,0,0,0.18);">
        <div style="font-weight: 700; margin-bottom: 6px; font-size: 12px; color: #2c3e50; border-bottom: 1px solid #e1e8ed; padding-bottom: 4px;">Legenda do Inventário</div>
        <div style="display: flex; align-items: center; margin-bottom: 5px;">
            <span style="background: #2ecc71; width: 13px; height: 13px; display: inline-block; border-radius: 3px; margin-right: 8px; border: 1px solid #27ae60;"></span>
            <span style="color: #2c3e50;"><b>RNA</b> (Regeneração)</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 5px;">
            <span style="background: #f39c12; width: 13px; height: 13px; display: inline-block; border-radius: 3px; margin-right: 8px; border: 1px solid #d68910;"></span>
            <span style="color: #2c3e50;"><b>Enriquecimento</b></span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 5px;">
            <span style="background: #e74c3c; width: 13px; height: 13px; display: inline-block; border-radius: 3px; margin-right: 8px; border: 1px solid #c0392b;"></span>
            <span style="color: #2c3e50;"><b>Plantio Direto</b></span>
        </div>
        <div style="display: flex; align-items: center; margin-top: 6px; padding-top: 5px; border-top: 1px dashed #d5dbdb;">
            <span style="border-top: 2px dashed #00ffff; width: 16px; display: inline-block; margin-right: 8px;"></span>
            <span style="color: #555; font-size: 10px;">Limite PES Charapucu</span>
        </div>
    </div>
    '''
    m.get_root().html.add_child(Element(legenda_html))

    folium.LayerControl(position="topright", collapsed=False).add_to(m)
    return m

In [ ]:
# [4] Mini-Dashboard Interativo com Painel de Filtros e Atualização Reativa

# Controles interativos (ipywidgets)
metodos_disponiveis = sorted(gdf_base["metodo"].unique().tolist())
acessos_disponiveis = ["Todos"] + sorted(gdf_base["acesso_fluvial"].unique().tolist())
portes_disponiveis = ["Todos"] + sorted(gdf_base["classe_area"].unique().tolist())

area_min_val = float(gdf_base["area_ha"].min())
area_max_val = float(gdf_base["area_ha"].max())

w_metodos = widgets.SelectMultiple(
    options=metodos_disponiveis,
    value=metodos_disponiveis,
    description="Métodos:",
    layout=widgets.Layout(width="260px", height="85px")
)

w_acesso = widgets.Dropdown(
    options=acessos_disponiveis,
    value="Todos",
    description="Acesso Fluvial:",
    layout=widgets.Layout(width="280px")
)

w_porte = widgets.Dropdown(
    options=portes_disponiveis,
    value="Todos",
    description="Porte:",
    layout=widgets.Layout(width="280px")
)

w_area_slider = widgets.FloatRangeSlider(
    value=[area_min_val, area_max_val],
    min=area_min_val,
    max=area_max_val,
    step=0.5,
    description="Área (ha):",
    continuous_update=False,
    layout=widgets.Layout(width="400px")
)

btn_reset = widgets.Button(
    description="Resetar Filtros",
    button_style="warning",
    icon="undo",
    layout=widgets.Layout(width="140px")
)

btn_exportar = widgets.Button(
    description="Exportar Seleção",
    button_style="success",
    icon="download",
    layout=widgets.Layout(width="160px")
)

# Painéis de saída
out_kpis = widgets.Output()
out_graficos = widgets.Output()
out_mapa = widgets.Output()
out_msg = widgets.Output()

def aplicar_filtros():
    gdf = gdf_base.copy()
    # Filtro de Métodos
    if w_metodos.value:
        gdf = gdf[gdf["metodo"].isin(w_metodos.value)]
    else:
        gdf = gdf.iloc[0:0]
    
    # Filtro de Acesso Fluvial
    if w_acesso.value != "Todos":
        gdf = gdf[gdf["acesso_fluvial"] == w_acesso.value]
    
    # Filtro de Porte
    if w_porte.value != "Todos":
        gdf = gdf[gdf["classe_area"] == w_porte.value]
    
    # Filtro de Área
    a_min, a_max = w_area_slider.value
    gdf = gdf[(gdf["area_ha"] >= a_min) & (gdf["area_ha"] <= a_max)]
    return gdf

def atualizar_dashboard(*args):
    gdf_f = aplicar_filtros()
    
    with out_kpis:
        clear_output(wait=True)
        display(HTML(gerar_cards_kpi(gdf_f, meta_ha=130.0)))
        
    with out_graficos:
        clear_output(wait=True)
        fig = gerar_graficos_analiticos(gdf_f)
        fig.show()
        
    with out_mapa:
        clear_output(wait=True)
        mapa = construir_mapa(gdf_f, gdf_parque)
        display(mapa)

def resetar_filtros(b):
    w_metodos.value = metodos_disponiveis
    w_acesso.value = "Todos"
    w_porte.value = "Todos"
    w_area_slider.value = [area_min_val, area_max_val]
    atualizar_dashboard()

def exportar_selecao(b):
    gdf_f = aplicar_filtros()
    with out_msg:
        clear_output(wait=True)
        if gdf_f.empty:
            print("⚠️ Nenhum talhão selecionado para exportação.")
            return
        dir_out = Path("outputs/vetores")
        dir_out.mkdir(parents=True, exist_ok=True)
        path_gpkg = dir_out / "selecao_corrente_dashboard.gpkg"
        path_csv = Path("outputs/relatorios/selecao_corrente_dashboard.csv")
        path_csv.parent.mkdir(parents=True, exist_ok=True)
        
        gdf_f.to_file(path_gpkg, layer="selecao_filtrada", driver="GPKG")
        gdf_f.drop(columns=["geometry"]).to_csv(path_csv, index=False, encoding="utf-8-sig")
        
        print(f"✅ Exportação concluída com sucesso! ({len(gdf_f)} talhões, {gdf_f['area_ha'].sum():.2f} ha)")
        print(f"   • Vetor GPKG: {path_gpkg}")
        print(f"   • Tabela CSV: {path_csv}")

# Conectar eventos
w_metodos.observe(atualizar_dashboard, names="value")
w_acesso.observe(atualizar_dashboard, names="value")
w_porte.observe(atualizar_dashboard, names="value")
w_area_slider.observe(atualizar_dashboard, names="value")
btn_reset.on_click(resetar_filtros)
btn_exportar.on_click(exportar_selecao)

# Layout dos Filtros
linha_botoes = widgets.HBox([btn_reset, btn_exportar])
coluna_filtros1 = widgets.VBox([w_metodos])
coluna_filtros2 = widgets.VBox([w_acesso, w_porte])
coluna_filtros3 = widgets.VBox([w_area_slider, linha_botoes])

painel_filtros = widgets.HBox(
    [coluna_filtros1, coluna_filtros2, coluna_filtros3],
    layout=widgets.Layout(
        background_color="#f8f9fa",
        border="1px solid #dee2e6",
        border_radius="8px",
        padding="15px",
        margin="10px 0 20px 0",
        justify_content="space-between"
    )
)

# Renderização Inicial
display(widgets.VBox([
    painel_filtros,
    out_msg,
    out_kpis,
    out_graficos,
    out_mapa
]))

atualizar_dashboard()